# Path Robotics Software Interview - Code Review

## Context

A software engineer's responsibilities have shifted significantly in the era of LLM coding agents. We're spending much more time reviewing code than writing it. Judging software quality is more important than ever!

This repository is the output of GitHub Copilot circa December 2025. It was given a programming problem we, Path Robotics, used to give candidates. We're asking you, the candidate, to review the code and provide feedback. Specifically, we're asking :arrow_down:

* Pretend this repository is a pull request made by someone you work with.
* It's your job to give constructive feedback to improve the solution.
* Everything is fair game! Consider things like:
  * Use of data structures
  * Adherence to software design principles
  * Flow control
  * Testing
* **Important:** There is no "right answer" we're expecting.
  * The goal here is to frame a conversation about software quality, not to find some hidden bug or "gotcha".

## Problem Statement (what we asked the agent)

The agent was asked to write a maze solver that could find a path from a start point (S) to an end point (E) in a 2 dimensional maze. Here's an example :arrow_down:

```
#########################
S #   #       #         #
# ### # # ##### ####### #
#   #   #           #   #
### # ############# # # #
# # # #   #       # # # #
# # ### # # ##### # # # #
# #     #   #     # # # #
# ########### ####### # #
# #   #     #   #     # #
# # # # # ##### # ##### #
#   # # #     #   #     #
##### # ##### ##### #####
#     #   #   #   # #   #
# ### # ### # # # # # # #
# #   # #   # # #   # # #
# ##### # ##### ####### #
#       #               E
#########################
```

It was given a single constraint: use python. Everything else was left up as a judgement call.

---

## How to use this notebook

Execute the cells in order:

1. **`maze_solver.py`** — defines the `MazeSolver` class.
2. **`main.py`** — runs the solver against several example mazes.
3. **`test_maze_solver.py`** — runs the unit tests.

## `maze_solver.py`

In [1]:
class MazeSolver:
    def __init__(self, maze_string):
        self.maze = maze_string.split("\n")
        self.start = None
        self.end = None
        self.path = []

    def find_empty_space(self):
        if len(self.maze) == 1:
            row = self.maze[0]
            for i, char in enumerate(row):
                if char == " ":
                    return i
        return -1

    def find_start_and_end(self):
        """Find S (start) and E (end) positions in maze"""
        for row_idx, row in enumerate(self.maze):
            for col_idx, char in enumerate(row):
                if char == "S":
                    self.start = (row_idx, col_idx)
                elif char == "E":
                    self.end = (row_idx, col_idx)

    def explore_maze(self):
        if not self.start or not self.end:
            self.find_start_and_end()

        if not self.start or not self.end:
            return None

        # Use a simple queue for breadth-first exploration
        queue = [(self.start, [self.start])]
        visited = set([self.start])

        directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]

        while queue:
            current, path = queue.pop(0)

            if current == self.end:
                self.path = path
                return path

            row, col = current

            for dr, dc in directions:
                new_row, new_col = row + dr, col + dc
                new_pos = (new_row, new_col)

                # Check bounds
                if 0 <= new_row < len(self.maze) and 0 <= new_col < len(
                    self.maze[new_row]
                ):
                    cell = self.maze[new_row][new_col]

                    # Can move to empty space or end, if not visited
                    if (cell == " " or cell == "E") and new_pos not in visited:
                        visited.add(new_pos)
                        queue.append((new_pos, path + [new_pos]))

        return None

    def solve(self):
        """Main solving method"""
        # Check if single row first
        if len(self.maze) == 1:
            result = self.find_empty_space()
            if result != -1:
                return f"Empty space found at position {result}"

        # Try exploring the maze
        result = self.explore_maze()
        if result:
            return f"Path found with {len(result)} steps: {result}"

        return "No solution found"

## `main.py`

In [2]:
def main():
    print("=== Single row ===")
    maze1 = "### ###"
    solver = MazeSolver(maze1)
    print(solver.solve())

    print("\n=== Simple turns ===")
    maze2 = """#######
#S    #
##### #
#     #
# #####
#    E#
#######"""
    solver2 = MazeSolver(maze2)
    print(solver2.solve())

    print("\n=== Maze has rooms ===")
    maze3 = """#########
#S      #
# ##### #
# #   # #
# # # # #
# # # # #
#   #  E#
#########"""
    solver3 = MazeSolver(maze3)
    print(solver3.solve())

    print("\n=== Complex turns ===")
    maze4 = """###########
#S        #
##### ### #
#   # #   #
# # # # ###
# # #   # #
# ### # # #
#     #  E#
###########"""
    solver4 = MazeSolver(maze4)
    print(solver4.solve())

    print("\n=== Deadends ===")
    maze5 = """#########
#S#     #
# # ### #
# #   # #
# ### # #
#     #E#
#########"""
    solver5 = MazeSolver(maze5)
    print(solver5.solve())


main()

=== Single row ===
Empty space found at position 3

=== Simple turns ===
Path found with 17 steps: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (3, 4), (3, 3), (3, 2), (3, 1), (4, 1), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)]

=== Maze has rooms ===
Path found with 12 steps: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (2, 7), (3, 7), (4, 7), (5, 7), (6, 7)]

=== Complex turns ===
Path found with 15 steps: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (6, 7), (7, 7), (7, 8), (7, 9)]

=== Deadends ===
Path found with 23 steps: [(1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (4, 5), (3, 5), (3, 4), (3, 3), (2, 3), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (2, 7), (3, 7), (4, 7), (5, 7)]


## `test_maze_solver.py`

In [4]:
import unittest


class TestMazeSolver(unittest.TestCase):
    def test_single_row_maze(self):
        maze = "### ###"
        solver = MazeSolver(maze)
        result = solver.find_empty_space()
        self.assertEqual(result, 3)

    def test_simple_hallway(self):
        maze = """#######
#S    #
##### #
#     #
# #####
#    E#
#######"""
        solver = MazeSolver(maze)
        path = solver.explore_maze()
        self.assertIsNotNone(path)
        self.assertEqual(path[0], (1, 1))  # Start
        self.assertEqual(path[-1], (5, 5))  # End

    def test_maze_with_rooms(self):
        maze = """#########
#S      #
# ##### #
# #   # #
# # # # #
# # # # #
#   #  E#
#########"""
        solver = MazeSolver(maze)
        path = solver.explore_maze()
        self.assertIsNotNone(path)
        self.assertTrue(len(path) > 0)

    def test_no_solution(self):
        """Test maze with no solution"""
        maze = """#####
#S# #
### #
#  E#
#####"""
        solver = MazeSolver(maze)
        path = solver.explore_maze()
        self.assertIsNone(path)


unittest.main(argv=[""], verbosity=2, exit=False)

test_maze_with_rooms (__main__.TestMazeSolver.test_maze_with_rooms) ... ok
test_no_solution (__main__.TestMazeSolver.test_no_solution)
Test maze with no solution ... ok
test_simple_hallway (__main__.TestMazeSolver.test_simple_hallway) ... ok
test_single_row_maze (__main__.TestMazeSolver.test_single_row_maze) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.006s

OK
